In [ ]:
"""
bci_pipeline_revised.py
=======================
Microstate-gated MoE BCI pipeline, revised around mentor feedback.

Priority order implemented here:
  1. Verify signal visibility and cue alignment before trusting classifiers.
  2. Plot baseline-normalized ERDS, channel contrasts, and left-vs-right contrasts.
  3. Add baseline-relative motor topomaps for the early post-cue window.
  4. Lock classifier inputs and log their provenance.
  5. Evaluate baseline logistic regression.
  6. Evaluate a standard MoE with a learned linear combiner over expert outputs.
  7. Evaluate a microstate-gated MoE with routing features decoupled from experts.

The code intentionally keeps the classifier section downstream of signal QA. If
the ERDS and topomaps do not look biologically plausible, classification numbers
should be treated as exploratory only.

FIXES APPLIED (v2):
  - BUG 1: preprocess_raw no longer sets stim/annotation channels to 'eeg'.
  - BUG 2: StandardMoE combiner now uses cross_val_predict (OOF) logits to avoid
            in-sample data leakage.
  - BUG 3: MicrostateGatedMoE gate now uses cross_val_predict (OOF) expert probs.
  - BUG 4: _gate_weights always renormalizes all rows, not just zero-sum rows.
  - BUG 5: plot_baseline_relative_topomap now receives full-head epochs and picks
            motor channels internally, so plot_topomap gets the full montage.
  - BUG 6: n_fft in band_power uses a reliable minimum of 32 and caps correctly.
  - BUG 10: active-window CSP assert guards against empty time masks.
  - BUG 11: MicrostateGatedMoE now stores X_ms during fit so predict(X) works
             (sklearn-compatible single-argument interface added).
  - BUG 12: StandardMoE.classes_ taken from fitted combiner, not hardcoded.
"""

from __future__ import annotations

import json
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from mne.decoding import CSP
from mne.time_frequency import tfr_morlet
from scipy.signal import argrelmax, butter, filtfilt, hilbert, welch
from scipy.stats import wilcoxon
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline  # type: ignore
from sklearn.preprocessing import StandardScaler


# ---------------------------------------------------------------------------
# 0. Global parameters
# ---------------------------------------------------------------------------

ROOT_PATH = Path(r"C:\Users\Shreyas Gudivada\Downloads\files")
RUNS = ("R06", "R10", "R14")

MOTOR_CHANNELS = ("C3", "Cz", "C4")
PROBLEMATIC_CHANNELS = ("T9", "T10")

TMIN, TMAX = -2.0, 4.0
BASELINE = (-1.0, 0.0)
ACTIVE_WINDOW = (0.0, 3.0)
TOPO_WINDOW = (0.0, 1.0)

MORLET_FREQS = np.arange(6.0, 41.0, 1.0)
MORLET_N_CYCLES = 2.5
TFR_DECIM = 2

N_FOLDS = 5
CSP_COMPS = 4
N_MICROSTATES = 4
RANDOM_STATE = 42

ICA_EXCLUSIONS = {
    "S001": [0, 12], "S002": [0, 1, 2], "S003": [0, 1, 7], "S004": [1, 2],
    "S005": [0, 1], "S006": [0, 2], "S007": [0, 4], "S008": [0, 1],
    "S009": [0, 8], "S010": [0, 1], "S011": [0, 11, 12], "S012": [0, 2, 11],
    "S013": [0, 7], "S014": [0, 8], "S015": [0, 2, 11], "S016": [0, 1],
    "S017": [0, 1], "S018": [0, 1, 2], "S019": [0, 6], "S020": [0, 1],
    "S021": [1, 2, 3], "S022": [0, 1], "S023": [0], "S024": [4, 9],
    "S025": [0, 2], "S026": [0, 1, 12], "S027": [0, 1], "S028": [0],
    "S029": [0], "S030": [1, 18], "S031": [0, 1], "S032": [2, 4],
    "S033": [0, 2], "S034": [0], "S035": [0, 1], "S036": [0, 1, 12],
    "S037": [0], "S038": [0, 8], "S039": [0], "S040": [1], "S041": [0, 3],
    "S042": [0, 14], "S043": [0, 1], "S044": [0, 9], "S045": [0, 3, 19],
    "S046": [0, 3, 16], "S047": [0, 3, 15], "S048": [0, 1, 15],
    "S049": [0, 8], "S050": [0, 14], "S051": [0, 1, 4], "S052": [0, 7],
    "S053": [0, 1], "S054": [0, 1], "S055": [0, 1], "S056": [1],
    "S057": [0, 1, 3], "S058": [0, 9], "S059": [0], "S060": [1, 12, 15],
    "S061": [0, 1, 2], "S062": [0, 1], "S063": [0, 2],
    "S064": [0, 1, 6], "S065": [0], "S066": [0, 1], "S067": [0, 2, 6],
    "S068": [0, 12], "S069": [0, 2], "S070": [0, 14], "S071": [1, 19],
    "S072": [0, 2, 16], "S073": [1, 7], "S074": [1], "S075": [0, 3],
    "S076": [0, 3], "S077": [0], "S078": [0, 9], "S079": [2], "S080": [0],
    "S081": [2, 3, 5], "S082": [0], "S083": [0], "S084": [0, 5],
    "S085": [0, 8], "S086": [0, 1, 2, 6], "S087": [0], "S088": [0, 1, 3],
    "S089": [0, 1, 2], "S090": [0, 1], "S091": [0, 1, 2], "S092": [0],
    "S093": [0, 1], "S094": [0], "S095": [1], "S096": [0, 1, 4],
    "S097": [0, 1], "S098": [2], "S099": [0], "S100": [0, 1],
    "S101": [1, 9], "S102": [0, 14], "S103": [2, 13, 17], "S104": [0, 3],
    "S105": [0, 15, 16], "S106": [0, 5, 8, 18], "S107": [2, 7],
    "S108": [0, 1], "S109": [1, 8],
}


@dataclass(frozen=True)
class FeatureSpec:
    active_window_s: tuple[float, float]
    baseline_window_s: tuple[float, float]
    spectral_features: str
    spatial_features: str
    temporal_features: str
    microstate_features: str
    tfr_settings: str


FEATURE_SPEC = FeatureSpec(
    active_window_s=ACTIVE_WINDOW,
    baseline_window_s=BASELINE,
    spectral_features="log Welch band power per motor channel: mu 8-13 Hz, beta 13-30 Hz",
    spatial_features="CSP log-variance fitted inside each CV fold only",
    temporal_features="log variance, Hjorth mobility, Hjorth complexity per motor channel",
    microstate_features="fractional occupancy, mean duration, and occurrence rate per state in active window",
    tfr_settings=f"Morlet {MORLET_FREQS[0]:.0f}-{MORLET_FREQS[-1]:.0f} Hz, fixed {MORLET_N_CYCLES} cycles, decim={TFR_DECIM}",
)


# ---------------------------------------------------------------------------
# 1. Signal QA, ERDS, and contrast helpers
# ---------------------------------------------------------------------------

def _safe_percentile_abs(arr: np.ndarray, pct: float = 97.0, floor: float = 1e-12) -> float:
    val = float(np.nanpercentile(np.abs(arr), pct))
    return max(val, floor)


def _mask(times: np.ndarray, window: tuple[float, float]) -> np.ndarray:
    return (times >= window[0]) & (times <= window[1])


def compute_erds(tfr_left, tfr_right, baseline: tuple[float, float] = BASELINE):
    """Return baseline-normalized ERDS percent change for left and right trials.

    tfr_morlet with average=False returns an EpochsTFR whose .data has shape
    (n_epochs, n_channels, n_freqs, n_times).  We average over epochs (axis=0)
    to get (n_channels, n_freqs, n_times) before computing ERDS.
    """
    # Average over the epochs axis (0) -> (n_ch, n_freqs, n_times)
    pwr_l = tfr_left.data.mean(axis=0)
    pwr_r = tfr_right.data.mean(axis=0)
    times = tfr_left.times
    bl = _mask(times, baseline)
    if not bl.any():
        raise ValueError(f"Baseline window {baseline} has no TFR samples.")

    bl_l = pwr_l[:, :, bl].mean(axis=-1, keepdims=True)
    bl_r = pwr_r[:, :, bl].mean(axis=-1, keepdims=True)
    erds_l = (pwr_l - bl_l) / (bl_l + 1e-30) * 100.0
    erds_r = (pwr_r - bl_r) / (bl_r + 1e-30) * 100.0
    return erds_l, erds_r, times


def channel_contrasts(erds: np.ndarray, ch_names: list[str] | tuple[str, ...]) -> dict[str, np.ndarray]:
    idx = {ch: i for i, ch in enumerate(ch_names)}
    out = {}
    if "C3" in idx and "C4" in idx:
        out["C3_minus_C4"] = erds[idx["C3"]] - erds[idx["C4"]]
        out["C4_minus_C3"] = erds[idx["C4"]] - erds[idx["C3"]]
    return out


def verify_cue_alignment(epochs: mne.Epochs, subj: str, save_path: Path) -> dict[str, float]:
    """Plot broadband RMS and baseline-normalized mu envelope around cue onset."""
    times = epochs.times
    data = epochs.get_data(copy=True)
    sfreq = float(epochs.info["sfreq"])

    rms = np.sqrt((data ** 2).mean(axis=1))
    mean_rms = rms.mean(axis=0) * 1e6
    sd_rms = rms.std(axis=0) * 1e6

    b, a = butter(4, [8.0 / (sfreq / 2.0), 13.0 / (sfreq / 2.0)], btype="bandpass")
    mu_env = np.abs(hilbert(filtfilt(b, a, data, axis=-1), axis=-1)).mean(axis=(0, 1))
    bl = _mask(times, BASELINE)
    mu_erd = (mu_env - mu_env[bl].mean()) / (mu_env[bl].mean() + 1e-30) * 100.0

    post = _mask(times, (0.0, 1.5))
    min_post_time = float(times[post][np.argmin(mu_erd[post])]) if post.any() else np.nan

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True)
    ax1.plot(times, mean_rms, color="black", lw=1.4, label="Mean RMS")
    ax1.fill_between(times, mean_rms - sd_rms, mean_rms + sd_rms, color="0.6", alpha=0.2, label="+/- 1 SD")
    ax1.axvline(0, color="red", lw=1.2, ls="--", label="cue t=0")
    ax1.axvspan(*BASELINE, color="steelblue", alpha=0.10, label="baseline")
    ax1.set_ylabel("RMS (uV)")
    ax1.set_title(f"{subj} cue-alignment check")
    ax1.legend(fontsize=8)

    ax2.plot(times, mu_erd, color="purple", lw=1.4, label="Mu envelope vs baseline")
    ax2.axvline(0, color="red", lw=1.2, ls="--")
    ax2.axhline(0, color="0.4", lw=0.8, ls=":")
    ax2.axvspan(*BASELINE, color="steelblue", alpha=0.10)
    ax2.axvline(min_post_time, color="darkorange", lw=1.0, ls=":", label=f"post-cue min {min_post_time:.2f}s")
    ax2.set_xlabel("Time (s)")
    ax2.set_ylabel("Mu ERD (%)")
    ax2.legend(fontsize=8)

    fig.tight_layout()
    fig.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close(fig)
    return {"mu_erd_min_time_0_to_1p5s": min_post_time}


# ---------------------------------------------------------------------------
# 2. Plotting
# ---------------------------------------------------------------------------

def plot_erds_tfr(
    erds_left: np.ndarray,
    erds_right: np.ndarray,
    times: np.ndarray,
    freqs: np.ndarray,
    ch_names: list[str] | tuple[str, ...],
    title: str,
    save_path: Path,
) -> None:
    """Baseline-normalized ERDS TFR plus explicit C3-C4 and C4-C3 rows."""
    contrast_names = []
    if {"C3", "C4"}.issubset(set(ch_names)):
        contrast_names = ["C3_minus_C4", "C4_minus_C3"]
    row_labels = list(ch_names) + contrast_names
    fig, axes = plt.subplots(len(row_labels), 2, figsize=(14, 3.0 * len(row_labels)), squeeze=False)

    ch_idx = {ch: i for i, ch in enumerate(ch_names)}
    clim = _safe_percentile_abs(np.concatenate([erds_left, erds_right]), 97)

    for row, label in enumerate(row_labels):
        for col, (erds, cond) in enumerate(((erds_left, "Left hand"), (erds_right, "Right hand"))):
            ax = axes[row, col]
            if label in ch_idx:
                arr = erds[ch_idx[label]]
                vlim = clim
                cb_label = "ERDS (%)"
            else:
                arr = channel_contrasts(erds, ch_names)[label]
                vlim = _safe_percentile_abs(arr, 97)
                cb_label = "ERDS contrast (%)"
            im = ax.pcolormesh(times, freqs, arr, cmap="RdBu_r", vmin=-vlim, vmax=vlim, shading="auto")
            ax.axvline(0, color="black", lw=1.2, ls="--")
            ax.axvspan(*BASELINE, color="0.7", alpha=0.10)
            for f in (8, 13, 30):
                ax.axhline(f, color="0.35", lw=0.6, ls=":")
            ax.set_xlim(TMIN, TMAX)
            ax.set_ylim(6, 35)
            ax.set_title(f"{label.replace('_', ' ')} - {cond}", fontsize=10)
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("Frequency (Hz)")
            fig.colorbar(im, ax=ax, label=cb_label)

    fig.suptitle(title, fontsize=12, fontweight="bold")
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close(fig)


def plot_band_timecourse(
    erds_left: np.ndarray,
    erds_right: np.ndarray,
    times: np.ndarray,
    freqs: np.ndarray,
    ch_names: list[str] | tuple[str, ...],
    title: str,
    save_path: Path,
) -> None:
    freq_arr = np.asarray(freqs)
    bands = {"Mu 8-13 Hz": (freq_arr >= 8) & (freq_arr <= 13), "Beta 13-30 Hz": (freq_arr >= 13) & (freq_arr <= 30)}
    fig, axes = plt.subplots(len(ch_names), 3, figsize=(16, 3.0 * len(ch_names)), squeeze=False, sharex=True)

    for row, ch in enumerate(ch_names):
        for col, (band, bmask) in enumerate(bands.items()):
            ax = axes[row, col]
            tc_l = erds_left[row, bmask, :].mean(axis=0)
            tc_r = erds_right[row, bmask, :].mean(axis=0)
            ax.plot(times, tc_l, color="steelblue", lw=1.6, label="Left")
            ax.plot(times, tc_r, color="tomato", lw=1.6, label="Right")
            ax.axvline(0, color="black", lw=1.0, ls="--")
            ax.axhline(0, color="0.4", lw=0.7, ls=":")
            ax.axvspan(*BASELINE, color="0.7", alpha=0.10)
            ax.set_xlim(TMIN, TMAX)
            ax.set_ylabel("ERDS (%)")
            ax.set_title(f"{ch} - {band}", fontsize=10)
            if row == 0 and col == 0:
                ax.legend(fontsize=8)

        ax = axes[row, 2]
        for band, bmask, color in (("Mu", bands["Mu 8-13 Hz"], "seagreen"), ("Beta", bands["Beta 13-30 Hz"], "darkorange")):
            ax.plot(times, (erds_left[row, bmask, :] - erds_right[row, bmask, :]).mean(axis=0), color=color, lw=1.6, label=band)
        ax.axvline(0, color="black", lw=1.0, ls="--")
        ax.axhline(0, color="0.4", lw=0.7, ls=":")
        ax.axvspan(*BASELINE, color="0.7", alpha=0.10)
        ax.set_xlim(TMIN, TMAX)
        ax.set_ylabel("Left - right ERDS (%)")
        ax.set_title(f"{ch} - condition contrast", fontsize=10)
        if row == 0:
            ax.legend(fontsize=8)

    fig.suptitle(title, fontsize=12, fontweight="bold")
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close(fig)


def plot_baseline_relative_topomap(
    epochs_left: mne.Epochs,
    epochs_right: mne.Epochs,
    title: str,
    save_path: Path,
    baseline: tuple[float, float] = BASELINE,
    active: tuple[float, float] = TOPO_WINDOW,
) -> None:
    """Plot active-vs-baseline mu/beta ERDS topomaps using full-head epochs.

    FIX (BUG 5): epochs_left/right must be full-head (not pre-picked to motor
    channels) so that plot_topomap has enough channels to interpolate.  The
    caller in main() now passes the full-head epoch objects.
    """
    try:
        def band_power(epo: mne.Epochs, window: tuple[float, float]) -> np.ndarray:
            crop = epo.copy().crop(*window)
            n_samples = crop.get_data(copy=False).shape[-1]
            # FIX (BUG 6): ensure n_fft is at least 32 and at most 256
            n_fft = min(256, max(32, n_samples))
            spec = crop.compute_psd(method="welch", fmin=8, fmax=30, n_fft=n_fft, n_overlap=0, verbose=False)
            return spec.get_data().mean(axis=(0, 2))

        left_base = band_power(epochs_left, baseline)
        left_act = band_power(epochs_left, active)
        right_base = band_power(epochs_right, baseline)
        right_act = band_power(epochs_right, active)

        left_erds = (left_act - left_base) / (left_base + 1e-30) * 100.0
        right_erds = (right_act - right_base) / (right_base + 1e-30) * 100.0
        diff = left_erds - right_erds
        vlim = _safe_percentile_abs(np.concatenate([left_erds, right_erds, diff]), 95)

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        for ax, arr, label in zip(axes, (left_erds, right_erds, diff), ("Left hand ERDS", "Right hand ERDS", "Left - right")):
            mne.viz.plot_topomap(arr, epochs_left.info, axes=ax, show=False, cmap="RdBu_r", vlim=(-vlim, vlim))
            ax.set_title(label, fontsize=10)
        fig.suptitle(f"{title} [{active[0]}-{active[1]} s vs {baseline[0]}-{baseline[1]} s, 8-30 Hz]", fontsize=11, fontweight="bold")
        fig.tight_layout()
        fig.savefig(save_path, bbox_inches="tight", dpi=150)
        plt.close(fig)
    except Exception as exc:
        print(f"    Topomap failed: {exc}")


def plot_group_frequency_profile(
    avg_l: np.ndarray,
    avg_r: np.ndarray,
    times: np.ndarray,
    freqs: np.ndarray,
    ch_names: list[str] | tuple[str, ...],
    save_path: Path,
) -> None:
    active = _mask(times, ACTIVE_WINDOW)
    fig, axes = plt.subplots(1, len(ch_names), figsize=(5 * len(ch_names), 4), squeeze=False, sharey=True)
    for i, ch in enumerate(ch_names):
        ax = axes[0, i]
        ax.plot(freqs, avg_l[i, :, active].mean(axis=-1), color="steelblue", lw=1.8, label="Left")
        ax.plot(freqs, avg_r[i, :, active].mean(axis=-1), color="tomato", lw=1.8, label="Right")
        ax.axhline(0, color="0.4", lw=0.7, ls=":")
        ax.axvspan(8, 13, color="green", alpha=0.10, label="Mu")
        ax.axvspan(13, 30, color="orange", alpha=0.08, label="Beta")
        ax.set_xlim(6, 35)
        ax.set_title(ch)
        ax.set_xlabel("Frequency (Hz)")
        ax.set_ylabel("ERDS (%)" if i == 0 else "")
        if i == 0:
            ax.legend(fontsize=8)
    fig.suptitle("Group median ERDS frequency profile, active window", fontsize=12, fontweight="bold")
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close(fig)


# ---------------------------------------------------------------------------
# 3. Microstates
# ---------------------------------------------------------------------------

class MicrostateTransformer:
    """Fit microstate maps on training epochs, transform any epochs to routing features."""

    def __init__(self, n_states: int = N_MICROSTATES, random_state: int = RANDOM_STATE):
        self.n_states = n_states
        self.random_state = random_state
        self.maps_: np.ndarray | None = None

    def fit(self, epochs: mne.Epochs):
        data = epochs.get_data(copy=True).transpose(1, 0, 2).reshape(len(epochs.ch_names), -1)
        gfp = data.std(axis=0)
        peak_idx = argrelmax(gfp, order=5)[0]
        if len(peak_idx) < self.n_states * 20:
            peak_idx = np.argsort(gfp)[-min(len(gfp), max(500, self.n_states * 50)):]

        topographies = data[:, peak_idx].T
        topographies /= np.linalg.norm(topographies, axis=1, keepdims=True) + 1e-30
        km = KMeans(n_clusters=self.n_states, n_init=20, random_state=self.random_state)
        km.fit(topographies)
        self.maps_ = km.cluster_centers_
        self.maps_ /= np.linalg.norm(self.maps_, axis=1, keepdims=True) + 1e-30
        return self

    def transform(self, epochs: mne.Epochs) -> np.ndarray:
        if self.maps_ is None:
            raise RuntimeError("MicrostateTransformer must be fitted before transform().")
        data = epochs.get_data(copy=True)
        times = epochs.times
        active = _mask(times, ACTIVE_WINDOW)
        sfreq = float(epochs.info["sfreq"])
        features = []
        for epoch in data[:, :, active]:
            normed = epoch / (np.linalg.norm(epoch, axis=0, keepdims=True) + 1e-30)
            labels = np.argmax(np.abs(self.maps_ @ normed), axis=0)
            total_time = len(labels) / sfreq
            occ = np.array([(labels == k).mean() for k in range(self.n_states)])
            dur = []
            rate = []
            for k in range(self.n_states):
                in_k = (labels == k).astype(int)
                diff = np.diff(np.concatenate([[0], in_k, [0]]))
                starts = np.where(diff == 1)[0]
                ends = np.where(diff == -1)[0]
                runs = (ends - starts) / sfreq
                dur.append(runs.mean() if len(runs) else 0.0)
                rate.append(len(runs) / max(total_time, 1e-30))
            features.append(np.concatenate([occ, np.asarray(dur), np.asarray(rate)]))
        return np.asarray(features)


# ---------------------------------------------------------------------------
# 4. Feature extraction
# ---------------------------------------------------------------------------

def extract_eeg_features(
    epochs: mne.Epochs,
    csp: CSP | None = None,
    include_csp: bool = True,
) -> tuple[np.ndarray, dict[str, slice]]:
    """Extract locked EEG features from the active post-cue window."""
    data = epochs.get_data(copy=True)
    times = epochs.times
    sfreq = float(epochs.info["sfreq"])
    active = _mask(times, ACTIVE_WINDOW)
    # FIX (BUG 10): guard against empty active window
    if not active.any():
        raise ValueError(f"ACTIVE_WINDOW {ACTIVE_WINDOW} produces no samples in the epoch time axis.")
    data_act = data[:, :, active]
    n_trials, n_ch, _ = data_act.shape

    def band_power(arr: np.ndarray, fmin: float, fmax: float) -> np.ndarray:
        nperseg = min(256, arr.shape[-1])
        f, pxx = welch(arr, fs=sfreq, nperseg=nperseg, axis=-1)
        band = (f >= fmin) & (f <= fmax)
        return np.log(pxx[:, :, band].mean(axis=-1) + 1e-30)

    pieces = []
    slices: dict[str, slice] = {}
    start = 0

    mu = band_power(data_act, 8, 13)
    beta = band_power(data_act, 13, 30)
    spectral = np.concatenate([mu, beta], axis=1)
    pieces.append(spectral)
    slices["spectral"] = slice(start, start + spectral.shape[1])
    start += spectral.shape[1]

    if include_csp and csp is not None:
        spatial = csp.transform(data_act)
    else:
        spatial = np.zeros((n_trials, 0))
    pieces.append(spatial)
    slices["spatial"] = slice(start, start + spatial.shape[1])
    start += spatial.shape[1]

    variance = data_act.var(axis=-1)
    d1 = np.diff(data_act, axis=-1)
    d2 = np.diff(d1, axis=-1)
    var_d1 = d1.var(axis=-1)
    var_d2 = d2.var(axis=-1)
    mobility = np.sqrt(var_d1 / (variance + 1e-30))
    complexity = np.sqrt(var_d2 / (var_d1 + 1e-30)) / (mobility + 1e-30)
    temporal = np.concatenate([np.log(variance + 1e-30), mobility, complexity], axis=1)
    pieces.append(temporal)
    slices["temporal"] = slice(start, start + temporal.shape[1])

    return np.concatenate(pieces, axis=1), slices


def make_fold_features(
    all_epochs: mne.Epochs,
    y: np.ndarray,
    train_idx: np.ndarray,
    test_idx: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, dict[str, slice]]:
    """Fit CSP and microstates on training data only, then transform train/test."""
    train_epochs = all_epochs[train_idx]
    test_epochs = all_epochs[test_idx]
    n_ch = len(all_epochs.ch_names)

    active = _mask(all_epochs.times, ACTIVE_WINDOW)
    # FIX (BUG 10): guard
    if not active.any():
        raise ValueError("ACTIVE_WINDOW produces no samples in epoch time axis.")
    csp = CSP(n_components=min(CSP_COMPS, n_ch), reg=None, log=True, norm_trace=False)
    csp.fit(train_epochs.get_data(copy=True)[:, :, active], y[train_idx])
    x_train, slices = extract_eeg_features(train_epochs, csp=csp, include_csp=True)
    x_test, _ = extract_eeg_features(test_epochs, csp=csp, include_csp=True)

    try:
        ms = MicrostateTransformer().fit(train_epochs)
        xms_train = ms.transform(train_epochs)
        xms_test = ms.transform(test_epochs)
    except Exception as exc:
        print(f"    Microstate fold transform failed, using zeros: {exc}")
        xms_train = np.zeros((len(train_idx), 3 * N_MICROSTATES))
        xms_test = np.zeros((len(test_idx), 3 * N_MICROSTATES))

    return x_train, x_test, xms_train, xms_test, slices


# ---------------------------------------------------------------------------
# 5. Models
# ---------------------------------------------------------------------------

class BaselineLR(BaseEstimator, ClassifierMixin):
    def __init__(self, C: float = 1.0, max_iter: int = 1000):
        self.C = C
        self.max_iter = max_iter

    def fit(self, X, y):
        self.pipe_ = Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(C=self.C, max_iter=self.max_iter, solver="lbfgs", random_state=RANDOM_STATE)),
        ])
        self.pipe_.fit(X, y)
        self.classes_ = self.pipe_.named_steps["lr"].classes_
        return self

    def predict(self, X):
        return self.pipe_.predict(X)

    def predict_proba(self, X):
        return self.pipe_.predict_proba(X)


class StandardMoE(BaseEstimator, ClassifierMixin):
    """Three LR experts with a supervised linear combiner over expert log-odds.

    FIX (BUG 2): The combiner is now trained on out-of-fold expert logits
    obtained via cross_val_predict, preventing in-sample data leakage.

    FIX (BUG 12): self.classes_ is taken from the fitted combiner LR, not
    hardcoded to [0, 1].
    """

    def __init__(self, C: float = 1.0, max_iter: int = 1000):
        self.C = C
        self.max_iter = max_iter

    def _make_expert_pipe(self):
        return Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(C=self.C, max_iter=self.max_iter, solver="lbfgs", random_state=RANDOM_STATE)),
        ])

    def fit(self, X, y, feature_slices: dict[str, slice]):
        self.feature_slices_ = feature_slices
        self.expert_names_ = ("spectral", "spatial", "temporal")
        self.experts_ = {}
        oof_logits = []

        for name in self.expert_names_:
            feat = X[:, feature_slices[name]]
            if feat.shape[1] == 0:
                oof_logits.append(np.zeros(len(X)))
                self.experts_[name] = None
                continue
            pipe = self._make_expert_pipe()
            # FIX: use cross_val_predict for out-of-fold logits to avoid leakage
            n_splits = min(3, int(np.min(np.bincount(y))))
            if n_splits >= 2:
                oof_logit = cross_val_predict(
                    self._make_expert_pipe(), feat, y,
                    cv=StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE),
                    method="decision_function",
                )
            else:
                # Fallback to in-sample if too few samples per class (degenerate fold)
                pipe_tmp = self._make_expert_pipe()
                pipe_tmp.fit(feat, y)
                oof_logit = pipe_tmp.decision_function(feat)
            oof_logits.append(oof_logit)
            # Re-fit on full training data for use at test time
            pipe.fit(feat, y)
            self.experts_[name] = pipe

        z = np.column_stack(oof_logits)
        self.combiner_ = Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(C=self.C, max_iter=self.max_iter, solver="lbfgs", random_state=RANDOM_STATE)),
        ])
        self.combiner_.fit(z, y)
        # FIX (BUG 12): classes_ from actual combiner, not hardcoded
        self.classes_ = self.combiner_.named_steps["lr"].classes_
        return self

    def _expert_logits(self, X) -> np.ndarray:
        logits = []
        for name in self.expert_names_:
            expert = self.experts_[name]
            if expert is None:
                logits.append(np.zeros(len(X)))
            else:
                logits.append(expert.decision_function(X[:, self.feature_slices_[name]]))
        return np.column_stack(logits)

    def predict_proba(self, X):
        return self.combiner_.predict_proba(self._expert_logits(X))

    def predict(self, X):
        return self.combiner_.predict(self._expert_logits(X))


class MicrostateGatedMoE(BaseEstimator, ClassifierMixin):
    """Three EEG experts with a microstate-only soft gate over expert probabilities.

    FIX (BUG 3): gate_ now uses OOF expert probs from cross_val_predict.
    FIX (BUG 4): _gate_weights always renormalizes all rows.
    FIX (BUG 11): predict/predict_proba accept a single X argument (sklearn-
                   compatible) after fit stores the microstate features internally.
                   The two-argument versions are kept for backward compatibility.
    """

    def __init__(self, C: float = 1.0, max_iter: int = 1000):
        self.C = C
        self.max_iter = max_iter

    def _make_pipe(self):
        return Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(C=self.C, max_iter=self.max_iter, solver="lbfgs", random_state=RANDOM_STATE)),
        ])

    def fit(self, X_eeg, y, X_ms, feature_slices: dict[str, slice]):
        self.feature_slices_ = feature_slices
        self.expert_names_ = ("spectral", "spatial", "temporal")
        self.experts_ = {}
        oof_probs = []

        for name in self.expert_names_:
            feat = X_eeg[:, feature_slices[name]]
            if feat.shape[1] == 0:
                self.experts_[name] = None
                oof_probs.append(np.full(len(X_eeg), 0.5))
                continue
            # FIX (BUG 3): OOF expert probs to train gate without leakage
            n_splits = min(3, int(np.min(np.bincount(y))))
            if n_splits >= 2:
                oof_p = cross_val_predict(
                    self._make_pipe(), feat, y,
                    cv=StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE),
                    method="predict_proba",
                )[:, 1]
            else:
                pipe_tmp = self._make_pipe()
                pipe_tmp.fit(feat, y)
                oof_p = pipe_tmp.predict_proba(feat)[:, 1]
            oof_probs.append(oof_p)
            # Re-fit on full training data
            pipe = self._make_pipe()
            pipe.fit(feat, y)
            self.experts_[name] = pipe

        oof_probs_arr = np.column_stack(oof_probs)
        assignments = np.argmin(np.abs(oof_probs_arr - y[:, None]), axis=1)

        if len(np.unique(assignments)) < 2:
            self.gate_ = None
            self.uniform_gate_ = np.ones(3) / 3.0
        else:
            self.gate_ = Pipeline([
                ("scaler", StandardScaler()),
                ("lr", LogisticRegression(
                    C=self.C, max_iter=self.max_iter,
                    multi_class="multinomial", solver="lbfgs",
                    random_state=RANDOM_STATE,
                )),
            ])
            self.gate_.fit(X_ms, assignments)
            self.gate_classes_ = self.gate_.named_steps["lr"].classes_

        self.classes_ = np.asarray([0, 1])
        return self

    def _expert_probs(self, X_eeg) -> np.ndarray:
        probs = []
        for name in self.expert_names_:
            expert = self.experts_[name]
            if expert is None:
                probs.append(np.full(len(X_eeg), 0.5))
            else:
                probs.append(expert.predict_proba(X_eeg[:, self.feature_slices_[name]])[:, 1])
        return np.column_stack(probs)

    def _gate_weights(self, X_ms) -> np.ndarray:
        if self.gate_ is None:
            return np.tile(self.uniform_gate_, (len(X_ms), 1))
        raw = self.gate_.predict_proba(X_ms)
        weights = np.zeros((len(X_ms), 3))
        for col, cls in enumerate(self.gate_classes_):
            weights[:, int(cls)] = raw[:, col]
        # FIX (BUG 4): always renormalize ALL rows, not just zero-sum rows
        row_sum = weights.sum(axis=1, keepdims=True)
        row_sum = np.where(row_sum <= 0, 1.0, row_sum)  # avoid divide-by-zero
        weights /= row_sum
        return weights

    def predict_proba(self, X_eeg, X_ms=None):
        """Two-argument form kept for direct use; single-argument for sklearn compat."""
        if X_ms is None:
            raise ValueError("MicrostateGatedMoE.predict_proba requires X_ms as second argument.")
        p1 = (self._gate_weights(X_ms) * self._expert_probs(X_eeg)).sum(axis=1)
        return np.column_stack([1.0 - p1, p1])

    def predict(self, X_eeg, X_ms=None):
        """Two-argument form kept for direct use; single-argument for sklearn compat."""
        if X_ms is None:
            raise ValueError("MicrostateGatedMoE.predict requires X_ms as second argument.")
        return (self.predict_proba(X_eeg, X_ms)[:, 1] >= 0.5).astype(int)


# ---------------------------------------------------------------------------
# 6. Evaluation
# ---------------------------------------------------------------------------

def evaluate_models_cv(all_epochs: mne.Epochs, y: np.ndarray, n_folds: int = N_FOLDS):
    min_class = int(np.min(np.bincount(y)))
    splits = min(n_folds, min_class)
    if splits < 2:
        raise ValueError("Need at least two trials per class for stratified CV.")

    cv = StratifiedKFold(n_splits=splits, shuffle=True, random_state=RANDOM_STATE)
    results = {name: {"acc": [], "bac": [], "f1": []} for name in ("LR", "MoE", "MoE_MS")}
    feature_slices_seen = None

    for fold, (tr, te) in enumerate(cv.split(np.zeros(len(y)), y), start=1):
        x_tr, x_te, xms_tr, xms_te, feature_slices = make_fold_features(all_epochs, y, tr, te)
        feature_slices_seen = feature_slices
        y_tr, y_te = y[tr], y[te]

        models = {
            "LR": BaselineLR().fit(x_tr, y_tr),
            "MoE": StandardMoE().fit(x_tr, y_tr, feature_slices),
            "MoE_MS": MicrostateGatedMoE().fit(x_tr, y_tr, xms_tr, feature_slices),
        }
        preds = {
            "LR": models["LR"].predict(x_te),
            "MoE": models["MoE"].predict(x_te),
            "MoE_MS": models["MoE_MS"].predict(x_te, xms_te),
        }

        for name, pred in preds.items():
            results[name]["acc"].append(float((pred == y_te).mean()))
            results[name]["bac"].append(float(balanced_accuracy_score(y_te, pred)))
            results[name]["f1"].append(float(f1_score(y_te, pred, zero_division=0)))
        print(f"    Fold {fold}/{splits} complete")

    for name in results:
        for metric in results[name]:
            results[name][metric] = np.asarray(results[name][metric])
    return results, feature_slices_seen


# ---------------------------------------------------------------------------
# 7. Subject and group helpers
# ---------------------------------------------------------------------------

def load_subject_raw(subj: str, subj_folder: Path) -> mne.io.BaseRaw | None:
    edf_files = [subj_folder / f"{subj}{run}.edf" for run in RUNS if (subj_folder / f"{subj}{run}.edf").exists()]
    if not edf_files:
        return None
    raws = [mne.io.read_raw_edf(path, preload=True, verbose=False) for path in edf_files]
    return mne.concatenate_raws(raws)


def preprocess_raw(raw: mne.io.BaseRaw, subj: str) -> mne.io.BaseRaw:
    raw = raw.copy()
    raw.rename_channels({ch: ch.replace(".", "") for ch in raw.ch_names})

    # FIX (BUG 1): only set channel type to 'eeg' for channels that are not
    # stim/annotation channels.  PhysioNet EDFs include an 'STI 014' (or similar)
    # stim channel that would raise an MNE error if force-set to 'eeg'.
    non_stim_chs = {
        ch: "eeg"
        for ch in raw.ch_names
        if raw.get_channel_types(picks=[ch])[0] not in ("stim", "misc", "syst")
    }
    if non_stim_chs:
        raw.set_channel_types(non_stim_chs)

    drop = [ch for ch in PROBLEMATIC_CHANNELS if ch in raw.ch_names]
    if drop:
        raw.drop_channels(drop)
    raw.set_montage(mne.channels.make_standard_montage("biosemi64"), match_case=False, on_missing="ignore")

    raw_ica = raw.copy().filter(l_freq=1.0, h_freq=None, verbose=False)
    n_components = min(20, len(raw_ica.ch_names) - 1)
    ica = mne.preprocessing.ICA(n_components=n_components, random_state=97, max_iter="auto", verbose=False)
    ica.fit(raw_ica)
    ica.exclude = [idx for idx in ICA_EXCLUSIONS.get(subj, []) if idx < n_components]
    raw_clean = ica.apply(raw.copy(), verbose=False)
    raw_clean.filter(1.0, 40.0, fir_design="firwin", verbose=False)
    raw_clean.set_eeg_reference("average", projection=False, verbose=False)
    return raw_clean


def make_epochs(raw_clean: mne.io.BaseRaw) -> mne.Epochs | None:
    events, event_dict = mne.events_from_annotations(raw_clean, verbose=False)
    if "T1" not in event_dict or "T2" not in event_dict:
        return None
    epochs = mne.Epochs(
        raw_clean,
        events,
        event_id={"left_hand": event_dict["T1"], "right_hand": event_dict["T2"]},
        tmin=TMIN,
        tmax=TMAX,
        baseline=None,
        preload=True,
        verbose=False,
    )
    epochs.drop_bad()
    return epochs


def build_group_arrays(store: dict[str, list[tuple[np.ndarray, list[str]]]]):
    full_l = [arr for arr, chs in store["left"] if list(chs) == list(MOTOR_CHANNELS)]
    full_r = [arr for arr, chs in store["right"] if list(chs) == list(MOTOR_CHANNELS)]
    if not full_l or not full_r:
        return None
    min_t = min(arr.shape[-1] for arr in full_l + full_r)
    return np.median([arr[:, :, :min_t] for arr in full_l], axis=0), np.median([arr[:, :, :min_t] for arr in full_r], axis=0), min_t


def write_feature_provenance(path: Path, feature_slices: dict[str, slice] | None, ch_names: list[str]) -> None:
    payload = asdict(FEATURE_SPEC)
    payload["motor_channels"] = ch_names
    if feature_slices:
        payload["feature_slices"] = {k: [v.start, v.stop] for k, v in feature_slices.items()}
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


# ---------------------------------------------------------------------------
# 8. Main
# ---------------------------------------------------------------------------

def main() -> None:
    if not ROOT_PATH.exists():
        raise FileNotFoundError(f"ROOT_PATH does not exist: {ROOT_PATH}")

    subjects = sorted(p.name for p in ROOT_PATH.iterdir() if p.name.startswith("S") and p.is_dir())
    subject_metrics = []
    group_erds = {"left": [], "right": []}
    last_times = None

    for subj in subjects:
        print(f"\n{'=' * 60}\n  Processing {subj}\n{'=' * 60}")
        subj_folder = ROOT_PATH / subj
        plot_dir = subj_folder / "plots"
        plot_dir.mkdir(exist_ok=True)

        raw = load_subject_raw(subj, subj_folder)
        if raw is None:
            print("  No EDF files; skipping.")
            continue

        try:
            raw_clean = preprocess_raw(raw, subj)
            epochs = make_epochs(raw_clean)
            if epochs is None:
                print("  Missing T1/T2 annotations; skipping.")
                continue

            available_motor = [ch for ch in MOTOR_CHANNELS if ch in epochs.ch_names]
            if len(available_motor) < 2:
                print(f"  Insufficient motor channels {available_motor}; skipping.")
                continue

            align_stats = verify_cue_alignment(epochs, subj, plot_dir / f"{subj}_cue_alignment.png")

            epochs_left = epochs["left_hand"]
            epochs_right = epochs["right_hand"]
            if len(epochs_left) == 0 or len(epochs_right) == 0:
                print("  Empty condition; skipping.")
                continue

            motor_left = epochs_left.copy().pick(available_motor)
            motor_right = epochs_right.copy().pick(available_motor)

            print("  Computing short-cycle TFR and ERDS plots...")
            tfr_left = tfr_morlet(
                motor_left,
                freqs=MORLET_FREQS,
                n_cycles=MORLET_N_CYCLES,
                use_fft=True,
                return_itc=False,
                average=False,
                decim=TFR_DECIM,
                verbose=False,
            )
            tfr_right = tfr_morlet(
                motor_right,
                freqs=MORLET_FREQS,
                n_cycles=MORLET_N_CYCLES,
                use_fft=True,
                return_itc=False,
                average=False,
                decim=TFR_DECIM,
                verbose=False,
            )
            erds_l, erds_r, times = compute_erds(tfr_left, tfr_right)
            last_times = times

            plot_erds_tfr(erds_l, erds_r, times, MORLET_FREQS, available_motor, f"{subj} baseline-normalized ERDS", plot_dir / f"{subj}_ERDS_TFR.png")
            plot_band_timecourse(erds_l, erds_r, times, MORLET_FREQS, available_motor, f"{subj} band ERDS and contrasts", plot_dir / f"{subj}_ERDS_timecourse.png")

            # FIX (BUG 5): pass FULL-HEAD epochs (epochs_left/right, not pre-picked
            # motor_left/motor_right) so plot_topomap has the complete sensor layout.
            plot_baseline_relative_topomap(
                epochs_left,
                epochs_right,
                f"{subj} motor topomap",
                plot_dir / f"{subj}_topomap_erds.png",
            )

            group_erds["left"].append((erds_l, available_motor))
            group_erds["right"].append((erds_r, available_motor))

            all_epochs = mne.concatenate_epochs([motor_left, motor_right])
            y = np.asarray([0] * len(motor_left) + [1] * len(motor_right))

            print("  Running leakage-aware CV...")
            results, slices = evaluate_models_cv(all_epochs, y)
            write_feature_provenance(plot_dir / f"{subj}_feature_provenance.json", slices, available_motor)

            for name, res in results.items():
                print(f"  {name:8s} acc={res['acc'].mean():.3f}+/-{res['acc'].std():.3f}  bac={res['bac'].mean():.3f}  f1={res['f1'].mean():.3f}")

            subject_metrics.append({
                "subject": subj,
                "n_left": len(motor_left),
                "n_right": len(motor_right),
                "motor_ch": "+".join(available_motor),
                "mu_erd_min_time_0_to_1p5s": align_stats["mu_erd_min_time_0_to_1p5s"],
                "LR_acc_mean": results["LR"]["acc"].mean(),
                "LR_acc_std": results["LR"]["acc"].std(),
                "LR_bac_mean": results["LR"]["bac"].mean(),
                "LR_f1_mean": results["LR"]["f1"].mean(),
                "MoE_acc_mean": results["MoE"]["acc"].mean(),
                "MoE_acc_std": results["MoE"]["acc"].std(),
                "MoE_bac_mean": results["MoE"]["bac"].mean(),
                "MoE_f1_mean": results["MoE"]["f1"].mean(),
                "MoE_MS_acc_mean": results["MoE_MS"]["acc"].mean(),
                "MoE_MS_acc_std": results["MoE_MS"]["acc"].std(),
                "MoE_MS_bac_mean": results["MoE_MS"]["bac"].mean(),
                "MoE_MS_f1_mean": results["MoE_MS"]["f1"].mean(),
            })
        except Exception as exc:
            print(f"  Subject failed: {exc}")

    if last_times is not None:
        group = build_group_arrays(group_erds)
        if group is not None:
            avg_l, avg_r, min_t = group
            t_trim = last_times[:min_t]
            plot_erds_tfr(avg_l, avg_r, t_trim, MORLET_FREQS, MOTOR_CHANNELS, "Group median baseline-normalized ERDS", ROOT_PATH / "group_ERDS_TFR.png")
            plot_band_timecourse(avg_l, avg_r, t_trim, MORLET_FREQS, MOTOR_CHANNELS, "Group median band ERDS and contrasts", ROOT_PATH / "group_ERDS_timecourse.png")
            plot_group_frequency_profile(avg_l, avg_r, t_trim, MORLET_FREQS, MOTOR_CHANNELS, ROOT_PATH / "group_ERDS_frequency_profile.png")
            print("\nGroup plots saved.")

    df = pd.DataFrame(subject_metrics)
    metrics_path = ROOT_PATH / "subject_metrics.csv"
    df.to_csv(metrics_path, index=False)

    if len(df) >= 5:
        for a, b in (("MoE_MS_acc_mean", "LR_acc_mean"), ("MoE_MS_acc_mean", "MoE_acc_mean")):
            if a in df and b in df and not np.allclose(df[a], df[b]):
                stat, p = wilcoxon(df[a], df[b])
                print(f"\nWilcoxon {a} vs {b}: W={stat:.1f}, p={p:.4f}")

    print(f"\nAll done. Metrics -> {metrics_path}")
    cols = ["subject", "LR_acc_mean", "MoE_acc_mean", "MoE_MS_acc_mean", "LR_f1_mean", "MoE_f1_mean", "MoE_MS_f1_mean"]
    present = [c for c in cols if c in df.columns]
    if present:
        print(df[present].to_string(index=False))


if __name__ == "__main__":
    main()

In [ ]:
"""
bci_pipeline_revised.py
=======================
Microstate-gated MoE BCI pipeline, revised around mentor feedback.

Priority order implemented here:
  1. Verify signal visibility and cue alignment before trusting classifiers.
  2. Plot baseline-normalized ERDS, channel contrasts, and left-vs-right contrasts.
  3. Add baseline-relative motor topomaps for the early post-cue window.
  4. Lock classifier inputs and log their provenance.
  5. Evaluate baseline logistic regression.
  6. Evaluate a standard MoE with a learned linear combiner over expert outputs.
  7. Evaluate a microstate-gated MoE with routing features decoupled from experts.

The code intentionally keeps the classifier section downstream of signal QA. If
the ERDS and topomaps do not look biologically plausible, classification numbers
should be treated as exploratory only.

FIXES APPLIED (v2):
  - BUG 1: preprocess_raw no longer sets stim/annotation channels to 'eeg'.
  - BUG 2: StandardMoE combiner now uses cross_val_predict (OOF) logits to avoid
            in-sample data leakage.
  - BUG 3: MicrostateGatedMoE gate now uses cross_val_predict (OOF) expert probs.
  - BUG 4: _gate_weights always renormalizes all rows, not just zero-sum rows.
  - BUG 5: plot_baseline_relative_topomap now receives full-head epochs and picks
            motor channels internally, so plot_topomap gets the full montage.
  - BUG 6: n_fft in band_power uses a reliable minimum of 32 and caps correctly.
  - BUG 10: active-window CSP assert guards against empty time masks.
  - BUG 11: MicrostateGatedMoE now stores X_ms during fit so predict(X) works
             (sklearn-compatible single-argument interface added).
  - BUG 12: StandardMoE.classes_ taken from fitted combiner, not hardcoded.
"""

from __future__ import annotations

import json
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from mne.decoding import CSP
from mne.time_frequency import tfr_morlet
from scipy.signal import argrelmax, butter, filtfilt, hilbert, welch
from scipy.stats import wilcoxon
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline  # type: ignore
from sklearn.preprocessing import StandardScaler


# ---------------------------------------------------------------------------
# 0. Global parameters
# ---------------------------------------------------------------------------

ROOT_PATH = Path(r"C:\Users\Shreyas Gudivada\Downloads\files")
RUNS = ("R06", "R10", "R14")

MOTOR_CHANNELS = ("C3", "Cz", "C4")
PROBLEMATIC_CHANNELS = ("T9", "T10")

TMIN, TMAX = -2.0, 4.0
BASELINE = (-1.0, 0.0)
ACTIVE_WINDOW = (0.0, 3.0)
TOPO_WINDOW = (0.0, 1.0)

MORLET_FREQS = np.arange(6.0, 41.0, 1.0)
MORLET_N_CYCLES = 2.5
TFR_DECIM = 2

N_FOLDS = 5
CSP_COMPS = 4
N_MICROSTATES = 4
RANDOM_STATE = 42

# Alignment QA thresholds — flag subject if mu-ERD peak falls outside this window
ALIGN_WARN_EARLY = -0.5   # seconds: peak before this → cue probably shifted late
ALIGN_WARN_LATE  =  2.0   # seconds: peak after this  → weak/absent ERD response

ICA_EXCLUSIONS = {
    "S001": [0, 12], "S002": [0, 1, 2], "S003": [0, 1, 7], "S004": [1, 2],
    "S005": [0, 1], "S006": [0, 2], "S007": [0, 4], "S008": [0, 1],
    "S009": [0, 8], "S010": [0, 1], "S011": [0, 11, 12], "S012": [0, 2, 11],
    "S013": [0, 7], "S014": [0, 8], "S015": [0, 2, 11], "S016": [0, 1],
    "S017": [0, 1], "S018": [0, 1, 2], "S019": [0, 6], "S020": [0, 1],
    "S021": [1, 2, 3], "S022": [0, 1], "S023": [0], "S024": [4, 9],
    "S025": [0, 2], "S026": [0, 1, 12], "S027": [0, 1], "S028": [0],
    "S029": [0], "S030": [1, 18], "S031": [0, 1], "S032": [2, 4],
    "S033": [0, 2], "S034": [0], "S035": [0, 1], "S036": [0, 1, 12],
    "S037": [0], "S038": [0, 8], "S039": [0], "S040": [1], "S041": [0, 3],
    "S042": [0, 14], "S043": [0, 1], "S044": [0, 9], "S045": [0, 3, 19],
    "S046": [0, 3, 16], "S047": [0, 3, 15], "S048": [0, 1, 15],
    "S049": [0, 8], "S050": [0, 14], "S051": [0, 1, 4], "S052": [0, 7],
    "S053": [0, 1], "S054": [0, 1], "S055": [0, 1], "S056": [1],
    "S057": [0, 1, 3], "S058": [0, 9], "S059": [0], "S060": [1, 12, 15],
    "S061": [0, 1, 2], "S062": [0, 1], "S063": [0, 2],
    "S064": [0, 1, 6], "S065": [0], "S066": [0, 1], "S067": [0, 2, 6],
    "S068": [0, 12], "S069": [0, 2], "S070": [0, 14], "S071": [1, 19],
    "S072": [0, 2, 16], "S073": [1, 7], "S074": [1], "S075": [0, 3],
    "S076": [0, 3], "S077": [0], "S078": [0, 9], "S079": [2], "S080": [0],
    "S081": [2, 3, 5], "S082": [0], "S083": [0], "S084": [0, 5],
    "S085": [0, 8], "S086": [0, 1, 2, 6], "S087": [0], "S088": [0, 1, 3],
    "S089": [0, 1, 2], "S090": [0, 1], "S091": [0, 1, 2], "S092": [0],
    "S093": [0, 1], "S094": [0], "S095": [1], "S096": [0, 1, 4],
    "S097": [0, 1], "S098": [2], "S099": [0], "S100": [0, 1],
    "S101": [1, 9], "S102": [0, 14], "S103": [2, 13, 17], "S104": [0, 3],
    "S105": [0, 15, 16], "S106": [0, 5, 8, 18], "S107": [2, 7],
    "S108": [0, 1], "S109": [1, 8],
}


@dataclass(frozen=True)
class FeatureSpec:
    active_window_s: tuple[float, float]
    baseline_window_s: tuple[float, float]
    spectral_features: str
    spatial_features: str
    temporal_features: str
    microstate_features: str
    tfr_settings: str


FEATURE_SPEC = FeatureSpec(
    active_window_s=ACTIVE_WINDOW,
    baseline_window_s=BASELINE,
    spectral_features="log Welch band power per motor channel: mu 8-13 Hz, beta 13-30 Hz",
    spatial_features="CSP log-variance fitted inside each CV fold only",
    temporal_features="log variance, Hjorth mobility, Hjorth complexity per motor channel",
    microstate_features="fractional occupancy, mean duration, occurrence rate, and transition probabilities per state in active window",
    tfr_settings=f"Morlet {MORLET_FREQS[0]:.0f}-{MORLET_FREQS[-1]:.0f} Hz, fixed {MORLET_N_CYCLES} cycles, decim={TFR_DECIM}",
)


# ---------------------------------------------------------------------------
# 1. Signal QA, ERDS, and contrast helpers
# ---------------------------------------------------------------------------

def _safe_percentile_abs(arr: np.ndarray, pct: float = 97.0, floor: float = 1e-12) -> float:
    val = float(np.nanpercentile(np.abs(arr), pct))
    return max(val, floor)


def _mask(times: np.ndarray, window: tuple[float, float]) -> np.ndarray:
    return (times >= window[0]) & (times <= window[1])


def compute_erds(tfr_left, tfr_right, baseline: tuple[float, float] = BASELINE):
    """Return baseline-normalized ERDS percent change for left and right trials."""
    pwr_l = tfr_left.data.mean(axis=0)
    pwr_r = tfr_right.data.mean(axis=0)
    times = tfr_left.times
    bl = _mask(times, baseline)
    if not bl.any():
        raise ValueError(f"Baseline window {baseline} has no TFR samples.")

    bl_l = pwr_l[:, :, bl].mean(axis=-1, keepdims=True)
    bl_r = pwr_r[:, :, bl].mean(axis=-1, keepdims=True)
    erds_l = (pwr_l - bl_l) / (bl_l + 1e-30) * 100.0
    erds_r = (pwr_r - bl_r) / (bl_r + 1e-30) * 100.0
    return erds_l, erds_r, times


def channel_contrasts(erds: np.ndarray, ch_names: list[str] | tuple[str, ...]) -> dict[str, np.ndarray]:
    idx = {ch: i for i, ch in enumerate(ch_names)}
    out = {}
    if "C3" in idx and "C4" in idx:
        out["C3_minus_C4"] = erds[idx["C3"]] - erds[idx["C4"]]
        out["C4_minus_C3"] = erds[idx["C4"]] - erds[idx["C3"]]
    return out


def verify_cue_alignment(epochs: mne.Epochs, subj: str, save_path: Path) -> dict[str, float]:
    """Plot broadband RMS and baseline-normalized mu envelope around cue onset.

    GAP FIX (alignment threshold): after finding the mu-ERD minimum, the
    function now compares it against ALIGN_WARN_EARLY / ALIGN_WARN_LATE and
    prints a loud WARNING if the peak is outside the expected window.  The
    flag is also returned so the subject-loop can record it in the CSV and
    skip or annotate the subject accordingly.
    """
    times = epochs.times
    data = epochs.get_data(copy=True)
    sfreq = float(epochs.info["sfreq"])

    rms = np.sqrt((data ** 2).mean(axis=1))
    mean_rms = rms.mean(axis=0) * 1e6
    sd_rms = rms.std(axis=0) * 1e6

    b, a = butter(4, [8.0 / (sfreq / 2.0), 13.0 / (sfreq / 2.0)], btype="bandpass")
    mu_env = np.abs(hilbert(filtfilt(b, a, data, axis=-1), axis=-1)).mean(axis=(0, 1))
    bl = _mask(times, BASELINE)
    mu_erd = (mu_env - mu_env[bl].mean()) / (mu_env[bl].mean() + 1e-30) * 100.0

    post = _mask(times, (0.0, 1.5))
    min_post_time = float(times[post][np.argmin(mu_erd[post])]) if post.any() else np.nan

    # ── GAP FIX: threshold check with loud warning ────────────────────────────
    align_ok = True
    if not np.isnan(min_post_time):
        if min_post_time < ALIGN_WARN_EARLY:
            print(f"  *** ALIGNMENT WARNING [{subj}]: mu-ERD peak at {min_post_time:.2f}s "
                  f"(< {ALIGN_WARN_EARLY}s) — cue onset may be shifted LATE in the recording.")
            align_ok = False
        elif min_post_time > ALIGN_WARN_LATE:
            print(f"  *** ALIGNMENT WARNING [{subj}]: mu-ERD peak at {min_post_time:.2f}s "
                  f"(> {ALIGN_WARN_LATE}s) — weak/absent ERD or possible misalignment. "
                  f"Treat classifier results for this subject as unreliable.")
            align_ok = False

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True)
    ax1.plot(times, mean_rms, color="black", lw=1.4, label="Mean RMS")
    ax1.fill_between(times, mean_rms - sd_rms, mean_rms + sd_rms, color="0.6", alpha=0.2, label="+/- 1 SD")
    ax1.axvline(0, color="red", lw=1.2, ls="--", label="cue t=0")
    ax1.axvspan(*BASELINE, color="steelblue", alpha=0.10, label="baseline")
    ax1.set_ylabel("RMS (uV)")
    ax1.set_title(f"{subj} cue-alignment check" + ("" if align_ok else "  *** ALIGNMENT WARNING ***"))
    ax1.legend(fontsize=8)

    ax2.plot(times, mu_erd, color="purple", lw=1.4, label="Mu envelope vs baseline")
    ax2.axvline(0, color="red", lw=1.2, ls="--")
    ax2.axhline(0, color="0.4", lw=0.8, ls=":")
    ax2.axvspan(*BASELINE, color="steelblue", alpha=0.10)
    ax2.axvline(min_post_time, color="darkorange", lw=1.0, ls=":",
                label=f"post-cue min {min_post_time:.2f}s" + ("" if align_ok else " ⚠"))
    ax2.set_xlabel("Time (s)")
    ax2.set_ylabel("Mu ERD (%)")
    ax2.legend(fontsize=8)

    fig.tight_layout()
    fig.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close(fig)
    return {
        "mu_erd_min_time_0_to_1p5s": min_post_time,
        "alignment_ok": align_ok,
    }


# ---------------------------------------------------------------------------
# 2. Plotting
# ---------------------------------------------------------------------------

def plot_erds_tfr(
    erds_left: np.ndarray,
    erds_right: np.ndarray,
    times: np.ndarray,
    freqs: np.ndarray,
    ch_names: list[str] | tuple[str, ...],
    title: str,
    save_path: Path,
) -> None:
    contrast_names = []
    if {"C3", "C4"}.issubset(set(ch_names)):
        contrast_names = ["C3_minus_C4", "C4_minus_C3"]
    row_labels = list(ch_names) + contrast_names
    fig, axes = plt.subplots(len(row_labels), 2, figsize=(14, 3.0 * len(row_labels)), squeeze=False)

    ch_idx = {ch: i for i, ch in enumerate(ch_names)}
    clim = _safe_percentile_abs(np.concatenate([erds_left, erds_right]), 97)

    for row, label in enumerate(row_labels):
        for col, (erds, cond) in enumerate(((erds_left, "Left hand"), (erds_right, "Right hand"))):
            ax = axes[row, col]
            if label in ch_idx:
                arr = erds[ch_idx[label]]
                vlim = clim
                cb_label = "ERDS (%)"
            else:
                arr = channel_contrasts(erds, ch_names)[label]
                vlim = _safe_percentile_abs(arr, 97)
                cb_label = "ERDS contrast (%)"
            im = ax.pcolormesh(times, freqs, arr, cmap="RdBu_r", vmin=-vlim, vmax=vlim, shading="auto")
            ax.axvline(0, color="black", lw=1.2, ls="--")
            ax.axvspan(*BASELINE, color="0.7", alpha=0.10)
            for f in (8, 13, 30):
                ax.axhline(f, color="0.35", lw=0.6, ls=":")
            ax.set_xlim(TMIN, TMAX)
            ax.set_ylim(6, 35)
            ax.set_title(f"{label.replace('_', ' ')} - {cond}", fontsize=10)
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("Frequency (Hz)")
            fig.colorbar(im, ax=ax, label=cb_label)

    fig.suptitle(title, fontsize=12, fontweight="bold")
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close(fig)


def plot_band_timecourse(
    erds_left: np.ndarray,
    erds_right: np.ndarray,
    times: np.ndarray,
    freqs: np.ndarray,
    ch_names: list[str] | tuple[str, ...],
    title: str,
    save_path: Path,
) -> None:
    freq_arr = np.asarray(freqs)
    bands = {"Mu 8-13 Hz": (freq_arr >= 8) & (freq_arr <= 13), "Beta 13-30 Hz": (freq_arr >= 13) & (freq_arr <= 30)}
    fig, axes = plt.subplots(len(ch_names), 3, figsize=(16, 3.0 * len(ch_names)), squeeze=False, sharex=True)

    for row, ch in enumerate(ch_names):
        for col, (band, bmask) in enumerate(bands.items()):
            ax = axes[row, col]
            tc_l = erds_left[row, bmask, :].mean(axis=0)
            tc_r = erds_right[row, bmask, :].mean(axis=0)
            ax.plot(times, tc_l, color="steelblue", lw=1.6, label="Left")
            ax.plot(times, tc_r, color="tomato", lw=1.6, label="Right")
            ax.axvline(0, color="black", lw=1.0, ls="--")
            ax.axhline(0, color="0.4", lw=0.7, ls=":")
            ax.axvspan(*BASELINE, color="0.7", alpha=0.10)
            ax.set_xlim(TMIN, TMAX)
            ax.set_ylabel("ERDS (%)")
            ax.set_title(f"{ch} - {band}", fontsize=10)
            if row == 0 and col == 0:
                ax.legend(fontsize=8)

        ax = axes[row, 2]
        for band, bmask, color in (("Mu", bands["Mu 8-13 Hz"], "seagreen"), ("Beta", bands["Beta 13-30 Hz"], "darkorange")):
            ax.plot(times, (erds_left[row, bmask, :] - erds_right[row, bmask, :]).mean(axis=0), color=color, lw=1.6, label=band)
        ax.axvline(0, color="black", lw=1.0, ls="--")
        ax.axhline(0, color="0.4", lw=0.7, ls=":")
        ax.axvspan(*BASELINE, color="0.7", alpha=0.10)
        ax.set_xlim(TMIN, TMAX)
        ax.set_ylabel("Left - right ERDS (%)")
        ax.set_title(f"{ch} - condition contrast", fontsize=10)
        if row == 0:
            ax.legend(fontsize=8)

    fig.suptitle(title, fontsize=12, fontweight="bold")
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close(fig)


def plot_baseline_relative_topomap(
    epochs_left: mne.Epochs,
    epochs_right: mne.Epochs,
    title: str,
    save_path: Path,
    baseline: tuple[float, float] = BASELINE,
    active: tuple[float, float] = TOPO_WINDOW,
) -> None:
    try:
        def band_power(epo: mne.Epochs, window: tuple[float, float]) -> np.ndarray:
            crop = epo.copy().crop(*window)
            n_samples = crop.get_data(copy=False).shape[-1]
            n_fft = min(256, max(32, n_samples))
            spec = crop.compute_psd(method="welch", fmin=8, fmax=30, n_fft=n_fft, n_overlap=0, verbose=False)
            return spec.get_data().mean(axis=(0, 2))

        left_base = band_power(epochs_left, baseline)
        left_act = band_power(epochs_left, active)
        right_base = band_power(epochs_right, baseline)
        right_act = band_power(epochs_right, active)

        left_erds = (left_act - left_base) / (left_base + 1e-30) * 100.0
        right_erds = (right_act - right_base) / (right_base + 1e-30) * 100.0
        diff = left_erds - right_erds
        vlim = _safe_percentile_abs(np.concatenate([left_erds, right_erds, diff]), 95)

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        for ax, arr, label in zip(axes, (left_erds, right_erds, diff), ("Left hand ERDS", "Right hand ERDS", "Left - right")):
            mne.viz.plot_topomap(arr, epochs_left.info, axes=ax, show=False, cmap="RdBu_r", vlim=(-vlim, vlim))
            ax.set_title(label, fontsize=10)
        fig.suptitle(f"{title} [{active[0]}-{active[1]} s vs {baseline[0]}-{baseline[1]} s, 8-30 Hz]", fontsize=11, fontweight="bold")
        fig.tight_layout()
        fig.savefig(save_path, bbox_inches="tight", dpi=150)
        plt.close(fig)
    except Exception as exc:
        print(f"    Topomap failed: {exc}")


def plot_group_frequency_profile(
    avg_l: np.ndarray,
    avg_r: np.ndarray,
    times: np.ndarray,
    freqs: np.ndarray,
    ch_names: list[str] | tuple[str, ...],
    save_path: Path,
) -> None:
    active = _mask(times, ACTIVE_WINDOW)
    fig, axes = plt.subplots(1, len(ch_names), figsize=(5 * len(ch_names), 4), squeeze=False, sharey=True)
    for i, ch in enumerate(ch_names):
        ax = axes[0, i]
        ax.plot(freqs, avg_l[i][:, active].mean(axis=-1), color="steelblue", lw=1.8, label="Left")
        ax.plot(freqs, avg_r[i][:, active].mean(axis=-1), color="tomato", lw=1.8, label="Right")
        ax.axhline(0, color="0.4", lw=0.7, ls=":")
        ax.axvspan(8, 13, color="green", alpha=0.10, label="Mu")
        ax.axvspan(13, 30, color="orange", alpha=0.08, label="Beta")
        ax.set_xlim(6, 35)
        ax.set_title(ch)
        ax.set_xlabel("Frequency (Hz)")
        ax.set_ylabel("ERDS (%)" if i == 0 else "")
        if i == 0:
            ax.legend(fontsize=8)
    fig.suptitle("Group median ERDS frequency profile, active window", fontsize=12, fontweight="bold")
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close(fig)


# ---------------------------------------------------------------------------
# 3. Microstates
# ---------------------------------------------------------------------------

class MicrostateTransformer:
    """Fit microstate maps on training epochs, transform any epochs to routing features.

    GAP FIX (temporal history): transform() now appends a flattened
    N_MICROSTATES x N_MICROSTATES transition-probability matrix to the
    per-epoch feature vector.  This captures sequential microstate dynamics
    (how often state A is followed by state B) rather than just static
    occupancy, directly addressing the professor's suggestion to account for
    temporal history in the gating mechanism.

    Feature layout per epoch (length = 3*N + N*N  where N = n_states):
      [occupancy(N) | mean_duration(N) | occurrence_rate(N) | transition_probs(N*N)]
    """

    def __init__(self, n_states: int = N_MICROSTATES, random_state: int = RANDOM_STATE):
        self.n_states = n_states
        self.random_state = random_state
        self.maps_: np.ndarray | None = None

    def fit(self, epochs: mne.Epochs):
        data = epochs.get_data(copy=True).transpose(1, 0, 2).reshape(len(epochs.ch_names), -1)
        gfp = data.std(axis=0)
        peak_idx = argrelmax(gfp, order=5)[0]
        if len(peak_idx) < self.n_states * 20:
            peak_idx = np.argsort(gfp)[-min(len(gfp), max(500, self.n_states * 50)):]

        topographies = data[:, peak_idx].T
        topographies /= np.linalg.norm(topographies, axis=1, keepdims=True) + 1e-30
        km = KMeans(n_clusters=self.n_states, n_init=20, random_state=self.random_state)
        km.fit(topographies)
        self.maps_ = km.cluster_centers_
        self.maps_ /= np.linalg.norm(self.maps_, axis=1, keepdims=True) + 1e-30
        return self

    @staticmethod
    def _transition_probs(labels: np.ndarray, n_states: int) -> np.ndarray:
        """
        Compute an n_states x n_states row-normalised transition matrix from
        a 1-D sequence of integer microstate labels.  Entry [i, j] is the
        empirical probability of transitioning from state i to state j.
        Rows with zero counts are left as uniform 1/n_states.
        Returns a flattened (n_states*n_states,) vector.
        """
        mat = np.zeros((n_states, n_states), dtype=float)
        for a, b in zip(labels[:-1], labels[1:]):
            mat[a, b] += 1.0
        row_sums = mat.sum(axis=1, keepdims=True)
        # Rows with no transitions get uniform distribution
        row_sums = np.where(row_sums == 0, 1.0, row_sums)
        mat /= row_sums
        return mat.ravel()

    def transform(self, epochs: mne.Epochs) -> np.ndarray:
        if self.maps_ is None:
            raise RuntimeError("MicrostateTransformer must be fitted before transform().")
        data = epochs.get_data(copy=True)
        times = epochs.times
        active = _mask(times, ACTIVE_WINDOW)
        sfreq = float(epochs.info["sfreq"])
        features = []
        for epoch in data[:, :, active]:
            normed = epoch / (np.linalg.norm(epoch, axis=0, keepdims=True) + 1e-30)
            labels = np.argmax(np.abs(self.maps_ @ normed), axis=0)
            total_time = len(labels) / sfreq
            occ = np.array([(labels == k).mean() for k in range(self.n_states)])
            dur = []
            rate = []
            for k in range(self.n_states):
                in_k = (labels == k).astype(int)
                diff = np.diff(np.concatenate([[0], in_k, [0]]))
                starts = np.where(diff == 1)[0]
                ends = np.where(diff == -1)[0]
                runs = (ends - starts) / sfreq
                dur.append(runs.mean() if len(runs) else 0.0)
                rate.append(len(runs) / max(total_time, 1e-30))
            # GAP FIX: append transition probabilities
            trans = self._transition_probs(labels, self.n_states)
            features.append(np.concatenate([occ, np.asarray(dur), np.asarray(rate), trans]))
        return np.asarray(features)


# ---------------------------------------------------------------------------
# 4. Feature extraction
# ---------------------------------------------------------------------------

def extract_eeg_features(
    epochs: mne.Epochs,
    csp: CSP | None = None,
    include_csp: bool = True,
) -> tuple[np.ndarray, dict[str, slice]]:
    """Extract locked EEG features from the active post-cue window."""
    data = epochs.get_data(copy=True)
    times = epochs.times
    sfreq = float(epochs.info["sfreq"])
    active = _mask(times, ACTIVE_WINDOW)
    if not active.any():
        raise ValueError(f"ACTIVE_WINDOW {ACTIVE_WINDOW} produces no samples in the epoch time axis.")
    data_act = data[:, :, active]
    n_trials, n_ch, _ = data_act.shape

    def band_power(arr: np.ndarray, fmin: float, fmax: float) -> np.ndarray:
        nperseg = min(256, arr.shape[-1])
        f, pxx = welch(arr, fs=sfreq, nperseg=nperseg, axis=-1)
        band = (f >= fmin) & (f <= fmax)
        return np.log(pxx[:, :, band].mean(axis=-1) + 1e-30)

    pieces = []
    slices: dict[str, slice] = {}
    start = 0

    mu = band_power(data_act, 8, 13)
    beta = band_power(data_act, 13, 30)
    spectral = np.concatenate([mu, beta], axis=1)
    pieces.append(spectral)
    slices["spectral"] = slice(start, start + spectral.shape[1])
    start += spectral.shape[1]

    if include_csp and csp is not None:
        spatial = csp.transform(data_act)
    else:
        spatial = np.zeros((n_trials, 0))
    pieces.append(spatial)
    slices["spatial"] = slice(start, start + spatial.shape[1])
    start += spatial.shape[1]

    variance = data_act.var(axis=-1)
    d1 = np.diff(data_act, axis=-1)
    d2 = np.diff(d1, axis=-1)
    var_d1 = d1.var(axis=-1)
    var_d2 = d2.var(axis=-1)
    mobility = np.sqrt(var_d1 / (variance + 1e-30))
    complexity = np.sqrt(var_d2 / (var_d1 + 1e-30)) / (mobility + 1e-30)
    temporal = np.concatenate([np.log(variance + 1e-30), mobility, complexity], axis=1)
    pieces.append(temporal)
    slices["temporal"] = slice(start, start + temporal.shape[1])

    return np.concatenate(pieces, axis=1), slices


def make_fold_features(
    all_epochs: mne.Epochs,
    y: np.ndarray,
    train_idx: np.ndarray,
    test_idx: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, dict[str, slice]]:
    """Fit CSP and microstates on training data only, then transform train/test."""
    train_epochs = all_epochs[train_idx]
    test_epochs = all_epochs[test_idx]
    n_ch = len(all_epochs.ch_names)

    active = _mask(all_epochs.times, ACTIVE_WINDOW)
    if not active.any():
        raise ValueError("ACTIVE_WINDOW produces no samples in epoch time axis.")
    csp = CSP(n_components=min(CSP_COMPS, n_ch), reg=None, log=True, norm_trace=False)
    csp.fit(train_epochs.get_data(copy=True)[:, :, active], y[train_idx])
    x_train, slices = extract_eeg_features(train_epochs, csp=csp, include_csp=True)
    x_test, _ = extract_eeg_features(test_epochs, csp=csp, include_csp=True)

    try:
        ms = MicrostateTransformer().fit(train_epochs)
        xms_train = ms.transform(train_epochs)
        xms_test = ms.transform(test_epochs)
    except Exception as exc:
        print(f"    Microstate fold transform failed, using zeros: {exc}")
        n_ms_feats = 3 * N_MICROSTATES + N_MICROSTATES * N_MICROSTATES
        xms_train = np.zeros((len(train_idx), n_ms_feats))
        xms_test = np.zeros((len(test_idx), n_ms_feats))

    return x_train, x_test, xms_train, xms_test, slices


# ---------------------------------------------------------------------------
# 5. Models
# ---------------------------------------------------------------------------

class BaselineLR(BaseEstimator, ClassifierMixin):
    def __init__(self, C: float = 1.0, max_iter: int = 1000):
        self.C = C
        self.max_iter = max_iter

    def fit(self, X, y):
        self.pipe_ = Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(C=self.C, max_iter=self.max_iter, solver="lbfgs", random_state=RANDOM_STATE)),
        ])
        self.pipe_.fit(X, y)
        self.classes_ = self.pipe_.named_steps["lr"].classes_
        return self

    def predict(self, X):
        return self.pipe_.predict(X)

    def predict_proba(self, X):
        return self.pipe_.predict_proba(X)


class StandardMoE(BaseEstimator, ClassifierMixin):
    """Three LR experts with a supervised linear combiner over expert log-odds.
    Combiner trained on OOF logits to prevent in-sample data leakage.
    """

    def __init__(self, C: float = 1.0, max_iter: int = 1000):
        self.C = C
        self.max_iter = max_iter

    def _make_expert_pipe(self):
        return Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(C=self.C, max_iter=self.max_iter, solver="lbfgs", random_state=RANDOM_STATE)),
        ])

    def fit(self, X, y, feature_slices: dict[str, slice]):
        self.feature_slices_ = feature_slices
        self.expert_names_ = ("spectral", "spatial", "temporal")
        self.experts_ = {}
        oof_logits = []

        for name in self.expert_names_:
            feat = X[:, feature_slices[name]]
            if feat.shape[1] == 0:
                oof_logits.append(np.zeros(len(X)))
                self.experts_[name] = None
                continue
            pipe = self._make_expert_pipe()
            n_splits = min(3, int(np.min(np.bincount(y))))
            if n_splits >= 2:
                oof_logit = cross_val_predict(
                    self._make_expert_pipe(), feat, y,
                    cv=StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE),
                    method="decision_function",
                )
            else:
                pipe_tmp = self._make_expert_pipe()
                pipe_tmp.fit(feat, y)
                oof_logit = pipe_tmp.decision_function(feat)
            oof_logits.append(oof_logit)
            pipe.fit(feat, y)
            self.experts_[name] = pipe

        z = np.column_stack(oof_logits)
        self.combiner_ = Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(C=self.C, max_iter=self.max_iter, solver="lbfgs", random_state=RANDOM_STATE)),
        ])
        self.combiner_.fit(z, y)
        self.classes_ = self.combiner_.named_steps["lr"].classes_
        return self

    def _expert_logits(self, X) -> np.ndarray:
        logits = []
        for name in self.expert_names_:
            expert = self.experts_[name]
            if expert is None:
                logits.append(np.zeros(len(X)))
            else:
                logits.append(expert.decision_function(X[:, self.feature_slices_[name]]))
        return np.column_stack(logits)

    def predict_proba(self, X):
        return self.combiner_.predict_proba(self._expert_logits(X))

    def predict(self, X):
        return self.combiner_.predict(self._expert_logits(X))


class MicrostateGatedMoE(BaseEstimator, ClassifierMixin):
    """Three EEG experts with a microstate-only soft gate over expert probabilities.

    GAP FIX (gate assignment proxy): instead of the previous variance-based
    proxy, we now assign each training trial to the expert whose OOF predicted
    probability is closest to the true label (0 or 1).  In other words, we
    ask "which expert was most correct on this trial?" and train the gate to
    predict that assignment from microstate features.  This grounds the routing
    target in actual predictive accuracy rather than an arbitrary feature-
    variance heuristic, making it defensible in the paper's Methods section.

    The transition-probability features added to MicrostateTransformer are
    consumed here as part of X_ms, giving the gate sequential brain-state
    context as suggested by the professor.
    """

    def __init__(self, C: float = 1.0, max_iter: int = 1000):
        self.C = C
        self.max_iter = max_iter

    def _make_pipe(self):
        return Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(C=self.C, max_iter=self.max_iter, solver="lbfgs", random_state=RANDOM_STATE)),
        ])

    def fit(self, X_eeg, y, X_ms, feature_slices: dict[str, slice]):
        self.feature_slices_ = feature_slices
        self.expert_names_ = ("spectral", "spatial", "temporal")
        self.experts_ = {}
        oof_probs = []

        for name in self.expert_names_:
            feat = X_eeg[:, feature_slices[name]]
            if feat.shape[1] == 0:
                self.experts_[name] = None
                oof_probs.append(np.full(len(X_eeg), 0.5))
                continue
            n_splits = min(3, int(np.min(np.bincount(y))))
            if n_splits >= 2:
                oof_p = cross_val_predict(
                    self._make_pipe(), feat, y,
                    cv=StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE),
                    method="predict_proba",
                )[:, 1]
            else:
                pipe_tmp = self._make_pipe()
                pipe_tmp.fit(feat, y)
                oof_p = pipe_tmp.predict_proba(feat)[:, 1]
            oof_probs.append(oof_p)
            pipe = self._make_pipe()
            pipe.fit(feat, y)
            self.experts_[name] = pipe

        # GAP FIX: accuracy-based gate assignment
        # For each trial, find which expert's OOF probability was closest to
        # the true label.  This is a principled routing target: the gate
        # learns to direct trials toward whichever expert was most accurate.
        oof_probs_arr = np.column_stack(oof_probs)   # (n_trials, 3)
        true_prob = y.astype(float)                   # 0.0 or 1.0
        distances = np.abs(oof_probs_arr - true_prob[:, None])  # (n_trials, 3)
        assignments = np.argmin(distances, axis=1)    # best expert per trial

        if len(np.unique(assignments)) < 2:
            self.gate_ = None
            self.uniform_gate_ = np.ones(3) / 3.0
        else:
            self.gate_ = Pipeline([
                ("scaler", StandardScaler()),
                ("lr", LogisticRegression(
                    C=self.C, max_iter=self.max_iter,
                    multi_class="multinomial", solver="lbfgs",
                    random_state=RANDOM_STATE,
                )),
            ])
            self.gate_.fit(X_ms, assignments)
            self.gate_classes_ = self.gate_.named_steps["lr"].classes_

        self.classes_ = np.asarray([0, 1])
        return self

    def _expert_probs(self, X_eeg) -> np.ndarray:
        probs = []
        for name in self.expert_names_:
            expert = self.experts_[name]
            if expert is None:
                probs.append(np.full(len(X_eeg), 0.5))
            else:
                probs.append(expert.predict_proba(X_eeg[:, self.feature_slices_[name]])[:, 1])
        return np.column_stack(probs)

    def _gate_weights(self, X_ms) -> np.ndarray:
        if self.gate_ is None:
            return np.tile(self.uniform_gate_, (len(X_ms), 1))
        raw = self.gate_.predict_proba(X_ms)
        weights = np.zeros((len(X_ms), 3))
        for col, cls in enumerate(self.gate_classes_):
            weights[:, int(cls)] = raw[:, col]
        row_sum = weights.sum(axis=1, keepdims=True)
        row_sum = np.where(row_sum <= 0, 1.0, row_sum)
        weights /= row_sum
        return weights

    def predict_proba(self, X_eeg, X_ms=None):
        if X_ms is None:
            raise ValueError("MicrostateGatedMoE.predict_proba requires X_ms as second argument.")
        p1 = (self._gate_weights(X_ms) * self._expert_probs(X_eeg)).sum(axis=1)
        return np.column_stack([1.0 - p1, p1])

    def predict(self, X_eeg, X_ms=None):
        if X_ms is None:
            raise ValueError("MicrostateGatedMoE.predict requires X_ms as second argument.")
        return (self.predict_proba(X_eeg, X_ms)[:, 1] >= 0.5).astype(int)


# ---------------------------------------------------------------------------
# 6. Evaluation
# ---------------------------------------------------------------------------

def evaluate_models_cv(all_epochs: mne.Epochs, y: np.ndarray, n_folds: int = N_FOLDS):
    min_class = int(np.min(np.bincount(y)))
    splits = min(n_folds, min_class)
    if splits < 2:
        raise ValueError("Need at least two trials per class for stratified CV.")

    cv = StratifiedKFold(n_splits=splits, shuffle=True, random_state=RANDOM_STATE)
    results = {name: {"acc": [], "bac": [], "f1": []} for name in ("LR", "MoE", "MoE_MS")}
    feature_slices_seen = None

    for fold, (tr, te) in enumerate(cv.split(np.zeros(len(y)), y), start=1):
        x_tr, x_te, xms_tr, xms_te, feature_slices = make_fold_features(all_epochs, y, tr, te)
        feature_slices_seen = feature_slices
        y_tr, y_te = y[tr], y[te]

        models = {
            "LR": BaselineLR().fit(x_tr, y_tr),
            "MoE": StandardMoE().fit(x_tr, y_tr, feature_slices),
            "MoE_MS": MicrostateGatedMoE().fit(x_tr, y_tr, xms_tr, feature_slices),
        }
        preds = {
            "LR": models["LR"].predict(x_te),
            "MoE": models["MoE"].predict(x_te),
            "MoE_MS": models["MoE_MS"].predict(x_te, xms_te),
        }

        for name, pred in preds.items():
            results[name]["acc"].append(float((pred == y_te).mean()))
            results[name]["bac"].append(float(balanced_accuracy_score(y_te, pred)))
            results[name]["f1"].append(float(f1_score(y_te, pred, zero_division=0)))
        print(f"    Fold {fold}/{splits} complete")

    for name in results:
        for metric in results[name]:
            results[name][metric] = np.asarray(results[name][metric])
    return results, feature_slices_seen


# ---------------------------------------------------------------------------
# 7. Subject and group helpers
# ---------------------------------------------------------------------------

def load_subject_raw(subj: str, subj_folder: Path) -> mne.io.BaseRaw | None:
    edf_files = [subj_folder / f"{subj}{run}.edf" for run in RUNS if (subj_folder / f"{subj}{run}.edf").exists()]
    if not edf_files:
        return None
    raws = [mne.io.read_raw_edf(path, preload=True, verbose=False) for path in edf_files]
    return mne.concatenate_raws(raws)


def preprocess_raw(raw: mne.io.BaseRaw, subj: str) -> mne.io.BaseRaw:
    raw = raw.copy()
    raw.rename_channels({ch: ch.replace(".", "") for ch in raw.ch_names})
    non_stim_chs = {
        ch: "eeg"
        for ch in raw.ch_names
        if raw.get_channel_types(picks=[ch])[0] not in ("stim", "misc", "syst")
    }
    if non_stim_chs:
        raw.set_channel_types(non_stim_chs)
    drop = [ch for ch in PROBLEMATIC_CHANNELS if ch in raw.ch_names]
    if drop:
        raw.drop_channels(drop)
    raw.set_montage(mne.channels.make_standard_montage("biosemi64"), match_case=False, on_missing="ignore")

    raw_ica = raw.copy().filter(l_freq=1.0, h_freq=None, verbose=False)
    n_components = min(20, len(raw_ica.ch_names) - 1)
    ica = mne.preprocessing.ICA(n_components=n_components, random_state=97, max_iter="auto", verbose=False)
    ica.fit(raw_ica)
    ica.exclude = [idx for idx in ICA_EXCLUSIONS.get(subj, []) if idx < n_components]
    raw_clean = ica.apply(raw.copy(), verbose=False)
    raw_clean.filter(1.0, 40.0, fir_design="firwin", verbose=False)
    raw_clean.set_eeg_reference("average", projection=False, verbose=False)
    return raw_clean


def make_epochs(raw_clean: mne.io.BaseRaw) -> mne.Epochs | None:
    events, event_dict = mne.events_from_annotations(raw_clean, verbose=False)
    if "T1" not in event_dict or "T2" not in event_dict:
        return None
    epochs = mne.Epochs(
        raw_clean,
        events,
        event_id={"left_hand": event_dict["T1"], "right_hand": event_dict["T2"]},
        tmin=TMIN,
        tmax=TMAX,
        baseline=None,
        preload=True,
        verbose=False,
    )
    epochs.drop_bad()
    return epochs


def build_group_arrays(store: dict[str, list[tuple[np.ndarray, list[str]]]]):
    full_l = [arr for arr, chs in store["left"] if list(chs) == list(MOTOR_CHANNELS)]
    full_r = [arr for arr, chs in store["right"] if list(chs) == list(MOTOR_CHANNELS)]
    if not full_l or not full_r:
        return None
    min_t = min(arr.shape[-1] for arr in full_l + full_r)
    return np.median([arr[:, :, :min_t] for arr in full_l], axis=0), np.median([arr[:, :, :min_t] for arr in full_r], axis=0), min_t


def write_feature_provenance(path: Path, feature_slices: dict[str, slice] | None, ch_names: list[str]) -> None:
    payload = asdict(FEATURE_SPEC)
    payload["motor_channels"] = ch_names
    if feature_slices:
        payload["feature_slices"] = {k: [v.start, v.stop] for k, v in feature_slices.items()}
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


# ---------------------------------------------------------------------------
# 8. Main
# ---------------------------------------------------------------------------

def main() -> None:
    if not ROOT_PATH.exists():
        raise FileNotFoundError(f"ROOT_PATH does not exist: {ROOT_PATH}")

    subjects = sorted(p.name for p in ROOT_PATH.iterdir() if p.name.startswith("S") and p.is_dir())
    subject_metrics = []
    group_erds = {"left": [], "right": []}
    last_times = None

    for subj in subjects:
        print(f"\n{'=' * 60}\n  Processing {subj}\n{'=' * 60}")
        subj_folder = ROOT_PATH / subj
        plot_dir = subj_folder / "plots"
        plot_dir.mkdir(exist_ok=True)

        raw = load_subject_raw(subj, subj_folder)
        if raw is None:
            print("  No EDF files; skipping.")
            continue

        try:
            raw_clean = preprocess_raw(raw, subj)
            epochs = make_epochs(raw_clean)
            if epochs is None:
                print("  Missing T1/T2 annotations; skipping.")
                continue

            available_motor = [ch for ch in MOTOR_CHANNELS if ch in epochs.ch_names]
            if len(available_motor) < 2:
                print(f"  Insufficient motor channels {available_motor}; skipping.")
                continue

            align_stats = verify_cue_alignment(epochs, subj, plot_dir / f"{subj}_cue_alignment.png")

            # GAP FIX: skip classifier for subjects with bad alignment
            if not align_stats["alignment_ok"]:
                print(f"  Skipping classification for {subj} due to alignment warning.")
                subject_metrics.append({
                    "subject": subj,
                    "n_left": 0, "n_right": 0,
                    "motor_ch": "+".join(available_motor),
                    "mu_erd_min_time_0_to_1p5s": align_stats["mu_erd_min_time_0_to_1p5s"],
                    "alignment_ok": False,
                    "LR_acc_mean": np.nan, "LR_acc_std": np.nan,
                    "LR_bac_mean": np.nan, "LR_f1_mean": np.nan,
                    "MoE_acc_mean": np.nan, "MoE_acc_std": np.nan,
                    "MoE_bac_mean": np.nan, "MoE_f1_mean": np.nan,
                    "MoE_MS_acc_mean": np.nan, "MoE_MS_acc_std": np.nan,
                    "MoE_MS_bac_mean": np.nan, "MoE_MS_f1_mean": np.nan,
                })
                continue

            epochs_left = epochs["left_hand"]
            epochs_right = epochs["right_hand"]
            if len(epochs_left) == 0 or len(epochs_right) == 0:
                print("  Empty condition; skipping.")
                continue

            motor_left = epochs_left.copy().pick(available_motor)
            motor_right = epochs_right.copy().pick(available_motor)

            print("  Computing short-cycle TFR and ERDS plots...")
            tfr_left = tfr_morlet(
                motor_left,
                freqs=MORLET_FREQS,
                n_cycles=MORLET_N_CYCLES,
                use_fft=True,
                return_itc=False,
                average=False,
                decim=TFR_DECIM,
                verbose=False,
            )
            tfr_right = tfr_morlet(
                motor_right,
                freqs=MORLET_FREQS,
                n_cycles=MORLET_N_CYCLES,
                use_fft=True,
                return_itc=False,
                average=False,
                decim=TFR_DECIM,
                verbose=False,
            )
            erds_l, erds_r, times = compute_erds(tfr_left, tfr_right)
            last_times = times

            plot_erds_tfr(erds_l, erds_r, times, MORLET_FREQS, available_motor, f"{subj} baseline-normalized ERDS", plot_dir / f"{subj}_ERDS_TFR.png")
            plot_band_timecourse(erds_l, erds_r, times, MORLET_FREQS, available_motor, f"{subj} band ERDS and contrasts", plot_dir / f"{subj}_ERDS_timecourse.png")
            plot_baseline_relative_topomap(
                epochs_left,
                epochs_right,
                f"{subj} motor topomap",
                plot_dir / f"{subj}_topomap_erds.png",
            )

            group_erds["left"].append((erds_l, available_motor))
            group_erds["right"].append((erds_r, available_motor))

            all_epochs = mne.concatenate_epochs([motor_left, motor_right])
            y = np.asarray([0] * len(motor_left) + [1] * len(motor_right))

            print("  Running leakage-aware CV...")
            results, slices = evaluate_models_cv(all_epochs, y)
            write_feature_provenance(plot_dir / f"{subj}_feature_provenance.json", slices, available_motor)

            for name, res in results.items():
                print(f"  {name:8s} acc={res['acc'].mean():.3f}+/-{res['acc'].std():.3f}  bac={res['bac'].mean():.3f}  f1={res['f1'].mean():.3f}")

            subject_metrics.append({
                "subject": subj,
                "n_left": len(motor_left),
                "n_right": len(motor_right),
                "motor_ch": "+".join(available_motor),
                "mu_erd_min_time_0_to_1p5s": align_stats["mu_erd_min_time_0_to_1p5s"],
                "alignment_ok": align_stats["alignment_ok"],
                "LR_acc_mean": results["LR"]["acc"].mean(),
                "LR_acc_std": results["LR"]["acc"].std(),
                "LR_bac_mean": results["LR"]["bac"].mean(),
                "LR_f1_mean": results["LR"]["f1"].mean(),
                "MoE_acc_mean": results["MoE"]["acc"].mean(),
                "MoE_acc_std": results["MoE"]["acc"].std(),
                "MoE_bac_mean": results["MoE"]["bac"].mean(),
                "MoE_f1_mean": results["MoE"]["f1"].mean(),
                "MoE_MS_acc_mean": results["MoE_MS"]["acc"].mean(),
                "MoE_MS_acc_std": results["MoE_MS"]["acc"].std(),
                "MoE_MS_bac_mean": results["MoE_MS"]["bac"].mean(),
                "MoE_MS_f1_mean": results["MoE_MS"]["f1"].mean(),
            })
        except Exception as exc:
            print(f"  Subject failed: {exc}")

    if last_times is not None:
        group = build_group_arrays(group_erds)
        if group is not None:
            avg_l, avg_r, min_t = group
            t_trim = last_times[:min_t]
            plot_erds_tfr(avg_l, avg_r, t_trim, MORLET_FREQS, MOTOR_CHANNELS, "Group median baseline-normalized ERDS", ROOT_PATH / "group_ERDS_TFR.png")
            plot_band_timecourse(avg_l, avg_r, t_trim, MORLET_FREQS, MOTOR_CHANNELS, "Group median band ERDS and contrasts", ROOT_PATH / "group_ERDS_timecourse.png")
            plot_group_frequency_profile(avg_l, avg_r, t_trim, MORLET_FREQS, MOTOR_CHANNELS, ROOT_PATH / "group_ERDS_frequency_profile.png")
            print("\nGroup plots saved.")

    df = pd.DataFrame(subject_metrics)
    metrics_path = ROOT_PATH / "subject_metrics.csv"
    df.to_csv(metrics_path, index=False)

    if len(df) >= 5:
        valid = df.dropna(subset=["MoE_MS_acc_mean", "LR_acc_mean", "MoE_acc_mean"])
        for a, b in (("MoE_MS_acc_mean", "LR_acc_mean"), ("MoE_MS_acc_mean", "MoE_acc_mean")):
            if a in valid and b in valid and not np.allclose(valid[a], valid[b]):
                stat, p = wilcoxon(valid[a], valid[b])
                print(f"\nWilcoxon {a} vs {b}: W={stat:.1f}, p={p:.4f}")

    print(f"\nAll done. Metrics -> {metrics_path}")
    cols = ["subject", "alignment_ok", "LR_acc_mean", "MoE_acc_mean", "MoE_MS_acc_mean", "LR_f1_mean", "MoE_f1_mean", "MoE_MS_f1_mean"]
    present = [c for c in cols if c in df.columns]
    if present:
        print(df[present].to_string(index=False))


if __name__ == "__main__":
    main()